In [11]:
import pandas as pd
import polars as pl
import numpy as np
import os
from pathlib import Path
import pandas as pd
import re, pathlib
import re

In [12]:
# =============================================================================
# 1.  Directory layout – pathlib all the way
# =============================================================================
SCRIPT_DIR   = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = SCRIPT_DIR.parent          # edit if your notebook is elsewhere

DATA_DIR       = PROJECT_ROOT / "data/"
SIMULATION_DIR = DATA_DIR / "simulations/"          # folder with ATTRIBUTE_* and wide SSP file
TORNADO_SIM_DIR = SIMULATION_DIR /"data_for_LSU"
OUTPUT_DIR     = DATA_DIR / "output/"

In [13]:
louisiana = pd.read_csv(TORNADO_SIM_DIR / "louisiana.csv")

In [ ]:
louisiana

In [ ]:
[c for c in louisiana.columns if c.startswith ("frac_trns_fuelmix_road_")]

In [14]:
# 1) Filter your base-case scenario
base_case = louisiana[louisiana["primary_id"] == 0].copy()

In [15]:


# 2) Grab the heavy‐duty + public‐transit vehicle‐km
dist_patterns = [
    r"^vehicle_distance_traveled_trns_road_heavy_.*$",
    r"^vehicle_distance_traveled_trns_public.*",
]
dist_cols = [c for c in base_case.columns if any(re.match(p, c) for p in dist_patterns)]
if not dist_cols:
    raise KeyError("No heavy‐duty/public distance columns found")
vehicle_distance = base_case[dist_cols].sum(axis=1)

# 3) Define your fuel‐mix fraction columns & read off baseline shares
segments = ["freight", "regional"]
relevant_fuels = [
    "biofuels", "diesel", "electricity",
    "gasoline", "hydrocarbon_gas_liquids",
    "hydrogen", "natural_gas"
]
frac_cols = {
    fuel: [f"frac_trns_fuelmix_road_heavy_{seg}_{fuel}" for seg in segments]
    for fuel in relevant_fuels
}
# make sure they all exist
for fuel, cols in frac_cols.items():
    for col in cols:
        if col not in base_case.columns:
            raise KeyError(f"Missing fraction column: {col}")

# baseline share at t=0 (sum of freight+regional)
frac_baseline = {
    fuel: base_case[cols].sum(axis=1).iloc[0]
    for fuel, cols in frac_cols.items()
}

# 4) Define your $/vkm multipliers for each fuel
multipliers = {
    "biofuels":             {"cost": 0.00,   "saving": 0.00},
    "diesel":               {"cost": 0.00,   "saving": 0.00},
    "electricity":          {"cost": 0.042,  "saving": 0.020},
    "gasoline":             {"cost": 0.00,   "saving": 0.00},
    "hydrocarbon_gas_liquids": {"cost":0.00, "saving": 0.00},
    "hydrogen":             {"cost": 0.00,   "saving": 0.00},
    "natural_gas":          {"cost": 0.00,   "saving": 0.00},
}

# 5) Compute current & baseline costs/savings per fuel
cost_now   = pd.Series(0.0, index=base_case.index)
cost_base  = pd.Series(0.0, index=base_case.index)
saving_now  = pd.Series(0.0, index=base_case.index)
saving_base = pd.Series(0.0, index=base_case.index)

for fuel in relevant_fuels:
    # current total vkm on this fuel
    cur_frac = base_case[frac_cols[fuel]].sum(axis=1)
    vkm_now  = vehicle_distance * cur_frac
    # baseline total vkm on this fuel
    vkm_base = vehicle_distance * frac_baseline[fuel]
    # accumulate
    cost_now   += vkm_now  * multipliers[fuel]["cost"]
    cost_base  += vkm_base * multipliers[fuel]["cost"]
    saving_now  += vkm_now  * multipliers[fuel]["saving"]
    saving_base += vkm_base * multipliers[fuel]["saving"]

# 6) Difference from baseline
cost_series   = cost_now   - cost_base
saving_series = saving_now - saving_base
net_series    = cost_series - saving_series

# 7) Build output
output_hd = pd.DataFrame({
    "primary_id":                          base_case["primary_id"],
    "region":                              base_case["region"],
    "time_period":                         base_case["time_period"],
    "vehicle_distance_traveled_total_vkm": vehicle_distance,
    "fuel_switch_cost_$":                  cost_series,
    "fuel_switch_savings_$":               saving_series,
    "fuel_switch_net_cost_$":              net_series,
}, index=base_case.index)


In [ ]:


# # ——————————————————————————————————————————
# # 2) Grab the heavy‐duty + public‐transit vehicle‐km
# # ——————————————————————————————————————————
# dist_patterns = [
#     r"^vehicle_distance_traveled_trns_road_heavy_.*$",
#     r"^vehicle_distance_traveled_trns_public.*",
# ]
# dist_cols = [
#     c for c in base_case.columns
#     if any(re.match(p, c) for p in dist_patterns)
# ]
# if not dist_cols:
#     raise KeyError("No heavy‐duty/public distance columns found")
# vehicle_distance = base_case[dist_cols].sum(axis=1)  # total vkm

# # ——————————————————————————————————————————
# # 3) Fuel‐mix fraction columns & baseline shares
# # ——————————————————————————————————————————
# segments = ["freight", "regional"]
# relevant_fuels = [
#     "biofuels","diesel","electricity",
#     "gasoline","hydrocarbon_gas_liquids",
#     "hydrogen","natural_gas"
# ]
# frac_cols = {
#     fuel: [
#         f"frac_trns_fuelmix_road_heavy_{seg}_{fuel}"
#         for seg in segments
#     ]
#     for fuel in relevant_fuels
# }
# # baseline share at t=0 (sum of freight+regional)
# frac_baseline = {
#     fuel: base_case[cols].sum(axis=1).iloc[0]
#     for fuel, cols in frac_cols.items()
# }

# # ——————————————————————————————————————————
# # 4) Compute switched‐to‐electricity vkm
# # ——————————————————————————————————————————
# elec_now   = base_case[frac_cols["electricity"]].sum(axis=1)
# switched_to_electricity = vehicle_distance * (
#     elec_now - frac_baseline["electricity"]
# )

# # ——————————————————————————————————————————
# # 5) Apply multipliers only to switched‐to‐electricity
# # ——————————————————————————————————————————
# elec_mult = {"cost":  0.042,   # $ per vkm
#              "saving":0.020}
# cost_series   = switched_to_electricity * elec_mult["cost"]
# saving_series = switched_to_electricity * elec_mult["saving"]
# net_series    = cost_series - saving_series  # fuel-switch net cost

# # ——————————————————————————————————————————
# # 6) Build the output DataFrame
# # ——————————————————————————————————————————
# output_hd = pd.DataFrame({
#     "primary_id":                          base_case["primary_id"],
#     "region":                              base_case["region"],
#     "time_period":                         base_case["time_period"],
#     "vehicle_distance_traveled_total_vkm": vehicle_distance,
#     "switched_to_electricity_vkm":         switched_to_electricity,
#     "fuel_switch_cost_$":                  cost_series,
#     "fuel_switch_savings_$":               saving_series,
#     "fuel_switch_net_cost_$":              net_series,
# }, index=base_case.index)






In [19]:
output_hd

,primary_id,region,time_period,vehicle_distance_traveled_total_vkm,fuel_switch_cost_$,fuel_switch_savings_$,fuel_switch_net_cost_$
0,0,louisiana,0,1.746209e+10,0.0,0.0,0.0
1,0,louisiana,1,1.670670e+10,0.0,0.0,0.0
2,0,louisiana,2,1.673614e+10,0.0,0.0,0.0
3,0,louisiana,3,1.769820e+10,0.0,0.0,0.0
4,0,louisiana,4,1.656701e+10,0.0,0.0,0.0
5,0,louisiana,5,1.831666e+10,0.0,0.0,0.0
6,0,louisiana,6,1.856498e+10,0.0,0.0,0.0
7,0,louisiana,7,1.884582e+10,0.0,0.0,0.0
8,0,louisiana,8,1.914999e+10,0.0,0.0,0.0
9,0,louisiana,9,1.947674e+10,0.0,0.0,0.0


In [20]:
output_hd = (
    output_hd
        .drop(
            columns=[
                'fuel_switch_cost_$',
                'fuel_switch_savings_$'
            ]
        )
        .rename(columns={
            'fuel_switch_net_cost_$': 'capex'
        })
)

In [21]:
output_hd

,primary_id,region,time_period,vehicle_distance_traveled_total_vkm,capex
0,0,louisiana,0,1.746209e+10,0.0
1,0,louisiana,1,1.670670e+10,0.0
2,0,louisiana,2,1.673614e+10,0.0
3,0,louisiana,3,1.769820e+10,0.0
4,0,louisiana,4,1.656701e+10,0.0
5,0,louisiana,5,1.831666e+10,0.0
6,0,louisiana,6,1.856498e+10,0.0
7,0,louisiana,7,1.884582e+10,0.0
8,0,louisiana,8,1.914999e+10,0.0
9,0,louisiana,9,1.947674e+10,0.0


In [22]:
# ——————————————————————————————————————————
# 7) (Optional) save to disk
# ——————————————————————————————————————————
OUTPUT_DIR = DATA_DIR / "output"
output_hd.to_csv(OUTPUT_DIR / "transportation_heavy_duty_fuel_switch_cost.csv", index=False)